In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("file1.csv")  # Make sure file1.csv is in the same folder
df.head()


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [ ]:
# Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("st", "state")
)

# Fix accidental replacement in "customer"
df.columns = [col.replace("custateomer", "customer") for col in df.columns]


In [ ]:
# Clean gender values
df['gender'] = df['gender'].replace({
    'F': 'F', 'Femal': 'F', 'female': 'F',
    'M': 'M', 'Male': 'M'
})

# Clean state names
df['state'] = df['state'].replace({
    'Cali': 'California',
    'AZ': 'Arizona',
    'WA': 'Washington'
})

# Standardize education names
df['education'] = df['education'].replace({
    'Bachelors': 'Bachelor'
})

# Remove % and convert customer lifetime value to numeric
df['customer_lifetime_value'] = df['customer_lifetime_value'].str.replace('%', '', regex=False)
df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')

# Group luxury vehicles
df['vehicle_class'] = df['vehicle_class'].replace({
    'Luxury SUV': 'Luxury',
    'Luxury Car': 'Luxury',
    'Sports Car': 'Luxury'
})


In [ ]:
# Extract middle number from complaint format like "1/5/00"
def extract_complaint_number(val):
    if isinstance(val, str) and '/' in val:
        parts = val.split('/')
        if len(parts) == 3:
            return parts[1]
    return val

df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(extract_complaint_number)
df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'], errors='coerce')

# Convert all numeric columns to integers (filling NaN with 0 first)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(0).astype(int)


In [ ]:
# Fill categorical columns with mode
for col in ['gender', 'state', 'education', 'policy_type', 'vehicle_class']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Drop rows with missing 'customer' value
df = df.dropna(subset=['customer'])


In [ ]:
# Drop duplicates and reset index
df = df.drop_duplicates().reset_index(drop=True)


In [ ]:
import pandas as pd

def clean_all(filepath):
    df = pd.read_csv(filepath)

    # Clean column names
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("st", "state")
    )
    df.columns = [col.replace("custateomer", "customer") for col in df.columns]

    # Clean values
    df['gender'] = df['gender'].replace({'F': 'F', 'Femal': 'F', 'female': 'F', 'M': 'M', 'Male': 'M'})
    df['state'] = df['state'].replace({'Cali': 'California', 'AZ': 'Arizona', 'WA': 'Washington'})
    df['education'] = df['education'].replace({'Bachelors': 'Bachelor'})
    df['customer_lifetime_value'] = df['customer_lifetime_value'].str.replace('%', '', regex=False)
    df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    df['vehicle_class'] = df['vehicle_class'].replace({'Luxury SUV': 'Luxury', 'Luxury Car': 'Luxury', 'Sports Car': 'Luxury'})

    # Fix data types
    def extract_complaint_number(val):
        if isinstance(val, str) and '/' in val:
            parts = val.split('/')
            if len(parts) == 3:
                return parts[1]
        return val

    df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(extract_complaint_number)
    df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'], errors='coerce')
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    df[numeric_cols] = df[numeric_cols].fillna(0).astype(int)

    # Handle nulls and duplicates
    for col in ['gender', 'state', 'education', 'policy_type', 'vehicle_class']:
        df[col] = df[col].fillna(df[col].mode()[0])
    df = df.dropna(subset=['customer'])
    df = df.drop_duplicates().reset_index(drop=True)

    return df


In [ ]:
# Define thresholds
claim_75 = df['total_claim_amount'].quantile(0.75)
clv_25 = df['customer_lifetime_value'].quantile(0.25)

# Filter customers
high_claim_low_value_df = df[
    (df['total_claim_amount'] > claim_75) &
    (df['customer_lifetime_value'] < clv_25)
]

# View summary statistics
high_claim_low_value_df[['total_claim_amount', 'customer_lifetime_value']].describe()
